# Monte Carlo pricing and equal-budget variance reduction

公开整理的数值研究 Notebook。研究无分红 GBM 下欧式看涨与单位现金二元看涨，比较抽样误差与离散方法。
这份说明重新编写，不包含考试题目、原提交报告和保存输出，也不需要行情数据。

默认使用小规模实验，当前运行产生当前结果；原 10 万路径实验没有在此重现。详见项目 README。

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
elif not (project_dir / "analysis.py").is_file():
    project_dir = project_dir / "projects" / "monte-carlo-pricing"
spec = importlib.util.spec_from_file_location("pricing", project_dir / "analysis.py")
pricing = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pricing
spec.loader.exec_module(pricing)

## 1. Parameters and analytical references

精确终值模拟从 GBM 终值分布直接抽样，仍有抽样误差。
与之不同，下面的 Black–Scholes 解析期权价格是确定性基准。
二元期权在终值严格高于执行价时支付 1；不在执行价相等时支付。

In [ ]:
parameters = pricing.Parameters()
PATHS = 20000  # Must be even for the equal-budget antithetic experiment.
STEPS = 64
print("Analytical call:", pricing.analytical_price(parameters))
print("Analytical unit-cash binary:", pricing.analytical_price(parameters, "binary"))

## 2. Three simulation methods

Euler 与 Milstein 的更新公式保留原学习实现，公开版改为按路径向量化并采用显式随机数流。
每种方法单独抽样，展示与解析价的差异、标准误、95% 正态近似区间与负终值路径数。
区间反映抽样误差，不包含离散偏差或模型风险；一次价格接近不证明离散误差为零。

In [ ]:
result = pricing.baseline(parameters, paths=PATHS, steps=STEPS, seed=42)
display(result[["method", "payoff", "price", "analytical_price", "difference_to_analytical",
                "standard_error", "ci_lower", "ci_upper", "negative_terminal_paths"]])

## 3. Antithetic pairs with the same total path budget

普通 N 条路径对应 N 个独立 payoff；N 条对偶路径只有 N/2 个独立 pair 均值。
估计量方差分别为样本方差除以 N 和 N/2，必须使用这一口径比较。
相同路径数不代表相同 CPU 成本；标准误降幅与运行时长分开显示。

In [ ]:
comparison, ratios = pricing.antithetic_comparison(parameters, total_paths=PATHS, steps=STEPS, seed=73)
display(comparison)
print("Equal-path-budget estimator variance ratio:", ratios["equal_path_budget_estimator_variance_ratio"])
print("Standard-error reduction fraction:", ratios["standard_error_reduction_fraction"])

## 4. Parameter scenarios

共 27 组单参数情景，使用每点 3,000 条路径的 Milstein 模拟，并对照解析曲线。
利率、波动率和期限的单调性只讨论此无分红、正利率参数范围；模拟曲线含抽样噪声。
利率上升会降低贴现因子，同时改变风险中性漂移，不能仅凭“贴现现值提高”解释看涨价格变化。

In [ ]:
scenarios = pricing.sensitivity(parameters, paths=3000, steps_per_year=64, seed=19)
pricing.plot_sensitivity(scenarios)
plt.show()
display(scenarios[["parameter", "value", "price", "standard_error", "analytical_price", "paths", "steps"]])

## Scope and next experiments

从项目目录运行 `python checks.py` 核对解析价、payoff 边界、零波动率、标准误和方差比。
公开版未运行原规模完整实验，未实测收敛阶，也没有生产性能承诺。
后续可以采用共同布朗增量、步长网格与多个随机种子，分离离散误差和抽样误差。
GBM 是简化模型；本项目没有交易回测、市场校准、随机波动率或路径依赖期权实现。